<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

*AQUIO VAN LOS NOMBRES DE TODOS*

*Aprendizaje Automático I — Prof. Milton Orlando Sarria Paja, PhD.*

----


# **Taller Práctico: Predicción de Productividad y Calidad de la Caña de Azúcar — Ingenio Providencia**

## 🎯 **Objetivos del taller**

El Ingenio Providencia cuenta con un conjunto de datos históricos sobre la producción de caña de azúcar. El objetivo de este trabajo es aplicar los conocimientos del curso para:

1. **Regresión** (`HISTORICO_SUERTES.xlsx`): predecir **TCH** (toneladas de caña por hectárea, indicador de productividad) y **%Sac.Caña** (porcentaje de sacarosa, indicador de calidad).
2. **Clasificación** (`BD_IPSA_1940.xlsx`): a partir de TCH y %Sac.Caña, clasificar los registros en niveles de desempeño (alto / medio / bajo).


📌 **Estructura de este notebook** (coincide con las 4 tareas del enunciado del taller):

1. Introducción y Contexto del Negocio
2. Análisis Exploratorio de Datos (EDA) y Preprocesamiento
3. Metodología de Modelamiento (Regresión y Clasificación)
4. Resultados y Discusión


---


## **1. Introducción y Contexto del Negocio (Tarea 1)**

📖 Antes de escribir una sola línea de código de modelado, es indispensable comprender el negocio: qué tan importante es el sector, qué factores agronómicos explican el rendimiento y la calidad de la caña, qué caracteriza al Ingenio Providencia, y qué se ha hecho antes con machine learning en este dominio. Esta contextualización guía decisiones posteriores de selección de variables, interpretación de resultados y priorización de hipótesis en el EDA (Tarea 2).


### 📌 **1.1 Importancia Económica y Social del sector azucarero (Valle del Cauca y Colombia)**

La agroindustria de la caña de azúcar es uno de los pilares económicos del Valle del Cauca:

- **Participación en el PIB:** el sector representa el **0,6% del PIB nacional** y el **2,4% del PIB agrícola del país**. A nivel regional, su peso es mucho mayor: **21,1% del PIB agrícola** y **10,2% del PIB industrial** del Valle del Cauca [1].
- **Generación de empleo:** la agroindustria genera aproximadamente **286.000 empleos** distribuidos en **50 municipios** de la región (cifra 2025) [1].
- **Número de ingenios:** operan **13 ingenios azucareros** en la región, de los cuales **6 cuentan con plantas de bioetanol** [1].
- **Producción y productividad:** en 2025 se molieron cerca de **23 millones de toneladas de caña** (+5% vs. 2024), produciendo cerca de **2 millones de toneladas de azúcar** [1][2]. El rendimiento promedio colombiano (~120 t/ha) **duplica el promedio mundial** (~60 t/ha) [2], aunque ha mostrado caídas recientes (de 127 a ~102-118 t/ha) por fenómenos climáticos (La Niña) e inseguridad regional (>5.000 ha comprometidas desde 2014) [2].

🧠 **¿Por qué importa esto para el taller?** Errores de pocos puntos porcentuales en la predicción de productividad, multiplicados por millones de toneladas, representan pérdidas económicas significativas para una industria que sostiene cientos de miles de empleos — esto justifica el esfuerzo de modelado que sigue en las Tareas 2-4.


### 📌 **1.2 Factores Clave del Rendimiento**

#### a) TCH (Toneladas de Caña por Hectárea)

- **Edad del cultivo y número de cortes:** en estudios de ML aplicados a caña de azúcar, el **número de cortes** aparece consistentemente como una de las variables con **mayor importancia predictiva** [4]. En nuestro dataset esto corresponde a `Edad Ult Cos`, `cortes` y `F.Ult.Corte`.
- **Clima:** el reciente descenso del rendimiento colombiano se explica en gran parte por un incremento del 25% en las precipitaciones respecto al promedio de 20 años (La Niña) [2] — esto valida la inclusión de las numerosas variables climáticas del dataset (`Lluvias Ciclo`, `Temp. Media Ciclo`, `Radiacion Solar Ciclo`, `Evaporacion Ciclo`, etc.).
- **Riego y tecnología:** Cenicaña ha impulsado sistemas de riego que reducen el consumo de agua en ~50%, sensores y programas de sostenibilidad (*Integra*) [2].
- **Suelo, variedad y distancia:** tipo de suelo, variedad sembrada y distancia al ingenio (`Suelo`, `Variedad`, `Dist Km`) son factores agronómicos y logísticos clásicos presentes en el dataset.
- **Estado del arte en modelado:** estudios recientes en el Valle del río Cauca muestran que **Random Forest, XGBoost y CatBoost superan consistentemente a la regresión lineal y a Ridge/Lasso** en la predicción de TCH [5] — antecedente relevante para interpretar el desempeño (limitado) de nuestros modelos lineales de referencia.

#### b) Porcentaje de Sacarosa (%Sac.Caña)

- **Variedad:** existen diferencias marcadas entre variedades "precoces" y "tardías" en cuanto a cuándo alcanzan su máximo contenido de sacarosa [3].
- **Maduración:** si el agua y el nitrógeno son abundantes, la planta **no madura** — de ahí la importancia agronómica de aplicar "madurantes" químicos en dosis controladas, exactamente lo que registran `Dosis Madurante` y `Semanas mad.` [3].
- **Radiación solar y temperatura:** la tasa fotosintética es óptima alrededor de **34°C**; temperaturas más altas durante la maduración disocian la sacarosa en fructosa y glucosa, reduciendo su acumulación. La radiación solar insuficiente también reduce la sacarosa [3][6].
- **Humedad:** la maduración requiere humedad relativa inferior al 65% [3].

🧠 **Implicación directa para la Tarea 2:** variables como `lluvias`, `Temp. Media Ciclo`, `Dosis Madurante`, `Semanas mad.` y `variedad` tienen fundamento agronómico sólido para ser predictoras relevantes de TCH y %Sac.Caña, y deben priorizarse en la selección de variables.


### 📌 **1.3 El Ingenio Providencia**

- **Ubicación y tamaño:** fundado en 1926, ubicado en **El Cerrito, Valle del Cauca**. Procesa **3,4 millones de toneladas de caña al año**, produciendo cerca de **270.000 toneladas de azúcar** y **67 millones de litros de alcohol** [7].
- **Participación de mercado:** produce el **15% del azúcar de Colombia** [7][8].
- **Mercado interno vs. exportación:** ~65% de su producción se queda en el mercado local; el resto se exporta a **35 países** [7].
- **Sostenibilidad:** es el **primer productor de azúcar orgánica de Colombia** [7] y alcanzó **autosostenibilidad energética** (250.951 MWh a partir del bagazo) [7].

🔹 **Pregunta de reflexión:** dado que Providencia combina producción convencional y orgánica, y exporta a mercados exigentes, ¿podrían existir sub-poblaciones distintas en el dataset (por `grupo_tenencia`, `variedad`, o prácticas de manejo) con perfiles agronómicos diferentes? Esto se explorará en el EDA (Tarea 2).


### 📌 **1.4 Estado del Arte: Machine Learning en Agricultura de Precisión**

- **Algoritmos más efectivos:** la evidencia reciente (incluyendo estudios colombianos en el Valle del río Cauca) indica que **Random Forest, XGBoost y CatBoost superan a la regresión lineal y a los métodos penalizados (Ridge/Lasso)** en precisión predictiva para TCH [4][5]. Esto no invalida el uso de modelos lineales en este taller (sirven como *benchmark* interpretable, como pide el enunciado), pero anticipa que su desempeño será un piso, no un techo — se discutirá como trabajo futuro.
- **Variables más predictivas:** el **número de cortes** aparece repetidamente como una de las variables de mayor importancia en modelos de ML para TCH [4]. Otros estudios usan **índices de percepción remota** (NDVI, MSI, evapotranspiración) como entradas.
- **Aplicación industrial (proceso, no solo campo):** el propio **Ingenio Providencia** reportó una **mejora del rendimiento del 10%** y una **reducción del consumo energético del 12%** tras implementar modelos analíticos (regresión lineal y árboles de decisión) sobre variables de proceso, siguiendo la metodología **CRISP-DM** [9] — evidencia de que la casa matriz del dataset ya tiene cultura analítica instalada.
- **Principal desafío reportado:** la **falta de datos de alta calidad** (faltantes, inconsistencias, variables no documentadas) es señalada como el principal obstáculo para aplicar ML en agricultura [4] — desafío que enfrentaremos directamente en la Tarea 2, dado que `HISTORICO_SUERTES.xlsx` tiene decenas de columnas climáticas/de insumos sin documentar en el diccionario oficial.

---

#### Fuentes citadas

[1] Asocaña / El País Cali, *"Caña de azúcar, el gran motor de la economía en el Valle del Cauca"*, 2025. https://www.asocana.org/modules/documentos/14167.aspx

[2] La República, *"Rendimiento de la caña de azúcar en Colombia duplica el promedio mundial"*, 2024. https://www.larepublica.co/economia/rendimiento-de-la-cana-de-azucar-en-colombia-duplica-el-promedio-mundial-3876015

[3] Cenicaña, *"Control y Características de Maduración"* / *"Calidad de la Caña de Azúcar"*, Libro El Cultivo de la Caña. https://www.cenicana.org/pdf_privado/documentos_no_seriados/libro_el_cultivo_cana/libro_p297-313.pdf

[4] ResearchGate, *"Predicción del rendimiento de cultivos agrícolas usando aprendizaje automático"*, 2021. https://www.researchgate.net/publication/349207652

[5] Universidad Nacional de Colombia, *"Predicción espacial del rendimiento del cultivo de caña de azúcar mediante aprendizaje de máquina"* (Valle del río Cauca). https://repositorio.unal.edu.co/items/34c4a06a-c916-4060-ad57-28e848712f2b

[6] Agencia de Noticias UNAL / AgroNET, *"Poca luz solar disminuye la sacarosa en el cultivo de caña de azúcar"*. https://agenciadenoticias.unal.edu.co/detalle/poca-luz-solar-disminuye-la-sacarosa-en-el-cultivo-de-cana-de-azucar

[7] Tecnicaña, *"Ingenio Providencia produce el 15% del azúcar en Colombia"*, 2024. https://tecnicana.org/2024/05/03/ingenios/ingenio-providencia/ingenio-providencia-produce-el-15-del-azucar-en-colombia/

[8] El País Cali, *"Providencia, primer productor de azúcar orgánica en Colombia"*. https://www.elpais.com.co/contenido/providencia-primer-productor-de-azucar-organica-en-colombia.html

[9] LinkedIn, José Rodríguez, *"De los datos al azúcar: cómo la analítica revoluciona..."* https://www.linkedin.com/pulse/de-los-datos-al-az%C3%BAcar-c%C3%B3mo-la-anal%C3%ADtica-revoluciona-jose-rodriguez-p2dqe/

---


## **2. Preparación del Entorno**

Antes de iniciar el Análisis Exploratorio de Datos (Tarea 2), importamos las librerías que usaremos a lo largo de todo el taller y hacemos una carga inicial de los dos datasets, únicamente para verificar que todo está disponible y en orden (dimensiones, tipos de datos). El EDA propiamente dicho (distribuciones, nulos, correlaciones, VIF, outliers) se desarrolla en la siguiente sección.

📌 **Nota de organización:** este notebook asume la siguiente estructura de carpetas (la misma del repositorio):

```
TallerPracticoAprendizajeAutom-tico1/
├── ArchivosImportantes/
│   ├── HISTORICO_SUERTES.xlsx
│   └── BD_IPSA_1940.xlsx
├── Diccionario/BD_cana.pdf
├── Notebooks/
│   └── Taller_Ingenio_Providencia.ipynb   <- este notebook
└── Informe_Avance.md
```


In [2]:
# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y modelado (se irán usando en las Tareas 2-4)
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier

# Métricas
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

# Estadística / diagnóstico
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


### 2.1 Carga de los datasets

Cargamos los dos archivos de Excel provistos por el Ingenio Providencia. Ambos están excluidos del repositorio Git (`.gitignore`) por tratarse de información de clientes/producción no autorizada para publicación — solo existen localmente.


In [3]:
# Dataset 1: HISTORICO_SUERTES -> tarea de REGRESIÓN (predecir TCH y %Sac.Caña)
df_hist = pd.read_excel("../ArchivosImportantes/HISTORICO_SUERTES.xlsx")

print("HISTORICO_SUERTES.xlsx")
print("Dimensiones (filas, columnas):", df_hist.shape)
df_hist.head()


HISTORICO_SUERTES.xlsx
Dimensiones (filas, columnas): (21027, 85)


,Período,Hacienda,Nombre,Zona,Tenencia,Suerte,Suelo,Area Neta,Dist Km,Variedad,Cod.Estado #,Cod.Estado,F.Siembra,D.S.,Ult.Riego,Edad Ult Cos,F.Ult.Corte,Destino 1=Semilla,Cod. T.Cultivo,Cultivo,Fec.Madur.,Producto,Dosis Madurante,Semanas mad.,TonUltCorte,TCH,TCHM,Ton.Azucar,Rdto,TAH,TAHM,Sac.Caña Precosecha,Edad.Precosecha,%Sac.Caña,%Sac.Muestreadora,%ATR,KATRHM,%Fibra Caña,%AR Jugo,%ME Min,%ME Veg,%ME Tot,Brix,Pureza,Vejez,Tipo Quema,T.Corte,Cerca de,Cosechó,Num.Riegos,M3 Riego,DDUlt.Riego,Lluvias (2 Meses Ant.),Lluvias Ciclo,Lluvias 0 -3,Lluvias tres a seis,Lluvias seis a nueve,Luvias 9 -FC,%Infest.Diatrea,Fosfato Jugo,Fert.Nitrogen.,Urea 46%,MEZ,Boro Granul.,MicroZinc,NITO_XTEND,Sul.Amonio,NITRAX-S,Vinaza,Codigo Estacion,Temp. Media 0-3,Temp. Media Ciclo,Temp Max Ciclo,Temp Min Ciclo,Humedad Rel Media 0-3,Humedad Rel Media Ciclo,Oscilacion Temp Med 0-3,Oscilacion Temp Ciclo,Sum Oscilacion Temp Ciclo,Radicion Solar 0-3,Radiacion Solar Ciclo,Precipitacion 0_3,Precipitacion Ciclo,Evaporacion 0-3,Evaporacion Ciclo
0,201701,80493,LA CONCHA,IP02,51.0,002A,CANTARINA,6.00,4.3,CC85-92,5,Corte 5,2010-08-20,NaN,NaT,12.81,2017-01-02,0,1.0,Normal,2016-11-04,BONUS 250 EC REGULADOR FISIOLÓGICO,1.0,8.428571,727.19,121.198333,9.461228,86.050,11.8332,14.341666,1.119567,16.8172,NaN,13.7582,13.508,15.1653,1434.823641,16.910,0.84,0.355,10.198,10.553,15.7464,87.2241,2.735,VERDE,MECANIZADO,El Cerrito,AI08,NaN,0.0,0,258.0,1038.0,0.0,454.0,102.0,482.0,NaN,178.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201701,81284,UKRANIA INCAUCA,IP05,81.0,039B,NaN,1.45,NaN,CC85-92,5,Corte 5,2011-01-27,NaN,NaT,11.14,2017-01-02,0,1.0,Normal,NaT,NaN,0.0,NaN,136.00,93.793103,8.419488,14.728,10.8294,10.157241,0.911781,NaN,NaN,12.8430,12.551,14.0410,1182.180399,16.936,0.55,2.298,7.273,9.571,15.2240,84.3602,73.823,Q.ACCIDENTAL,MANUAL,Candelaria,AI08,NaN,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,382.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201701,80203,EL AMPARO SAA,IP05,31.0,007,CORINTIAS,8.24,23.0,CC01-1228,4,Corte 4,2011-10-25,1.65,2016-09-17,12.32,2017-01-02,0,1.0,Normal,2016-11-04,BONUS 250 EC REGULADOR FISIOLÓGICO,1.1,8.428571,1436.62,174.347087,14.151549,145.268,10.1117,17.629611,1.430974,14.7749,12.02,11.9364,11.940,13.1236,1857.192723,15.512,0.61,3.000,9.323,12.323,14.1130,84.4527,2.108,VERDE,MECANIZADO,Palmira,AI08,5.0,48513.6,107,246.0,1002.0,106.0,326.0,113.0,457.0,NaN,226.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,201701,81380,SAN JUDAS INCAUCA,IP05,82.0,013A,NaN,1.05,66.5,CC01-1940,2,Corte 2,2014-03-08,NaN,NaT,9.79,2017-01-02,0,1.0,Normal,NaT,NaN,0.0,NaN,143.63,136.790476,13.972469,13.517,9.4109,12.873333,1.314947,NaN,NaN,11.2770,10.931,12.4820,1744.043640,17.621,0.67,0.140,6.788,6.927,13.6350,82.7062,64.614,Q.ACCIDENTAL,MANUAL,Corinto,AI08,NaN,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,278.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,201701,80298,JAVA,IP06,31.0,025A,GALPON,4.53,17.0,RB73-2223,3,Corte 3,2013-01-10,1.65,NaT,11.53,2017-01-02,0,1.0,Normal,NaT,NaN,0.0,NaN,512.20,113.068432,9.806455,42.505,8.2985,9.383002,0.813790,16.7662,NaN,10.2160,10.294,11.6030,1137.843039,14.352,0.95,0.592,2.939,3.531,12.9760,78.7299,71.021,Q.ACCIDENTAL,MANUAL,Guacari,AI08,NaN,0.0,0,138.0,991.0,264.0,255.0,188.0,284.0,NaN,244.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Dataset 2: BD_IPSA_1940 -> tarea de CLASIFICACIÓN (niveles de TCH y %Sac.Caña)
df_ipsa = pd.read_excel("../ArchivosImportantes/BD_IPSA_1940.xlsx", sheet_name="BD_IPSA")

print("BD_IPSA_1940.xlsx")
print("Dimensiones (filas, columnas):", df_ipsa.shape)
df_ipsa.head()


BD_IPSA_1940.xlsx
Dimensiones (filas, columnas): (2187, 21)


,Unnamed: 0,NOME,FAZ,TAL,tipocorte,variedad,madurada,producto,dosismad,semsmad,edad,cortes,me,vejez,sacarosa,mes,periodo,TCH,lluvias,grupo_tenencia,pct_diatrea
0,11,AMAIME SILCA,81291,40,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,8.3,12.3,4,12.7,2.4,14.0,12,202012,112,137,3,6.2
1,12,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.3,11.2,2,7.8,2.3,13.0,3,201903,157,0,3,3.5
2,13,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,7.9,12.2,3,8.8,1.8,13.3,3,202003,167,68,3,4.3
3,15,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.6,13.1,1,6.1,2.5,13.4,3,201903,156,0,3,3.5
4,16,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,8.1,12.2,2,7.9,2.1,14.0,3,202003,151,68,3,4.3


📌 **Verificación rápida (sanity check):** antes de pasar a la Tarea 2, confirmamos que las columnas clave mencionadas en la Tarea 1 (`TCH`, `%Sac.Caña`, `Edad Ult Cos`, `Variedad`, `lluvias`, etc.) efectivamente existen en los datasets cargados.


In [5]:
print("Columnas de HISTORICO_SUERTES relacionadas con lo investigado en la Tarea 1:")
cols_interes_hist = [c for c in df_hist.columns if any(
    k.lower() in c.lower() for k in ["TCH", "Sac", "Edad", "Variedad", "Lluvia", "Temp", "Dosis", "Suelo", "Dist"]
)]
print(cols_interes_hist)

print("\nColumnas de BD_IPSA_1940:")
print(df_ipsa.columns.tolist())


Columnas de HISTORICO_SUERTES relacionadas con lo investigado en la Tarea 1:
['Suelo', 'Dist Km', 'Variedad', 'Edad Ult Cos', 'Dosis Madurante', 'TCH', 'TCHM', 'Sac.Caña Precosecha', 'Edad.Precosecha', '%Sac.Caña', '%Sac.Muestreadora', 'Lluvias (2 Meses Ant.)', 'Lluvias Ciclo', 'Lluvias 0 -3', 'Lluvias tres a seis', 'Lluvias seis a nueve', 'Temp. Media 0-3', 'Temp. Media Ciclo', 'Temp Max Ciclo', 'Temp Min Ciclo', 'Humedad Rel Media 0-3 ', 'Humedad Rel Media Ciclo', 'Oscilacion Temp Med 0-3', 'Oscilacion Temp Ciclo', 'Sum Oscilacion Temp Ciclo']

Columnas de BD_IPSA_1940:
['Unnamed: 0', 'NOME', 'FAZ', 'TAL', 'tipocorte', 'variedad', 'madurada', 'producto', 'dosismad', 'semsmad', 'edad', 'cortes', 'me', 'vejez', 'sacarosa', 'mes', 'periodo', 'TCH', 'lluvias', 'grupo_tenencia', 'pct_diatrea']


---

## **3. Análisis Exploratorio de Datos (EDA) y Preprocesamiento (Tarea 2)**

*(Próximo paso — pendiente de desarrollar: selección y justificación de variables, análisis de valores faltantes, estrategia de imputación, distribuciones, correlaciones, detección de multicolinealidad con VIF)*

---

## **4. Metodología de Modelamiento (Tarea 3)**

*(Pendiente: regresión lineal múltiple, regresión logística multinomial, KNN, regularización, validación hold-out + k-fold CV)*

---

## **5. Resultados y Discusión (Tarea 4)**

*(Pendiente: tablas comparativas de desempeño, interpretación con base en el contexto de negocio, visualizaciones de apoyo)*
